# Lab 30: Unsupervised Threat Hunting

*AI Cybersecurity Masterclass · Lab 30 · AI60secLab*

**Companion video:** none yet for this lab.

**Level:** Practitioner

**Research question:** Without any labels, can we find attacks by learning what normal looks like, and where should an analyst spend a limited review budget?

Every earlier lab used labels to train. Real hunts rarely have them. Here, no model ever sees a label: we learn a baseline of normal traffic, score a new "hunt week" for unusual flows, group the week into behavior clusters, and only at the end reveal the labels to grade the hunt.

**Learning objectives**
- build a realistic hunt window where attacks are rare (1%);
- learn "normal" from a clean baseline with an Isolation Forest;
- measure precision at a fixed analyst budget, not just AUC;
- see why *unusual* and *malicious* are different things;
- see what happens when the baseline itself is contaminated;
- use clustering to decide where to look first.

## Security foundation: anomaly detection and hunting without labels

**Anomaly detection** flags activity that differs from a learned baseline. It needs no labels, so it can surface attacks nobody has seen before. Its weakness is built in: rare is not the same as bad. Backups at 3 a.m., a new monitoring tool, or one engineer's odd workflow are all unusual and all benign.

Two rules decide whether anomaly-based hunting works:
- **the baseline must be clean.** If attackers were already active while "normal" was learned, their behavior becomes part of normal;
- **attacks must be rare in the window.** Anomaly methods assume the odd thing out is the interesting thing. In UNSW-NB15's raw test set, most rows are attacks, which turns that assumption upside down. So this lab builds a realistic window with 1% attacks.

Hunters therefore combine anomaly scores with **behavioral grouping** (clusters of similar flows) and **hypotheses** ("UDP conversations that never complete, repeated to the same service"), then review a fixed number of leads per shift.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/v2
Data folder : /home/claude/run/v2/data/raw


In [2]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.hunting import (
    realistic_mix,
    fit_anomaly_model,
    anomaly_scores,
    precision_at_k,
    cluster_window,
    cluster_profile,
)

train, test = load_unsw(DATA_DIR)

# A clean baseline week (normal traffic only) and a hunt week with 1% attacks.
baseline = realistic_mix(train, attack_share=0.00, n_rows=30000, random_state=0)
hunt_week = realistic_mix(test, attack_share=0.01, n_rows=30000, random_state=1)

X_base, _ = split_xy(baseline, "label")
X_hunt, _ = split_xy(hunt_week, "label")
X_base, X_hunt = align_columns(X_base, X_hunt)
X_base = drop_identifier_like_columns(X_base)
X_hunt = drop_identifier_like_columns(X_hunt)

print("Baseline flows:", len(X_base), "| hunt-week flows:", len(X_hunt))
print("Labels used for fitting: none. They stay hidden until the grading step.")

Baseline flows: 30000 | hunt-week flows: 30000
Labels used for fitting: none. They stay hidden until the grading step.


## Step 1: Learn normal, score the hunt week

In [3]:
anomaly_model = fit_anomaly_model(X_base)
hunt_scores = anomaly_scores(anomaly_model, X_hunt)

hunt_week["anomaly_score"] = hunt_scores
display(hunt_week.sort_values("anomaly_score", ascending=False)
        [["proto", "service", "state", "spkts", "dpkts", "sbytes", "dbytes", "anomaly_score"]]
        .head(10).round(3))

,proto,service,state,spkts,dpkts,sbytes,dbytes,anomaly_score
23003,tcp,-,REQ,4,0,180,0,0.153
29195,tcp,-,REQ,4,0,180,0,0.153
1700,tcp,-,REQ,4,0,180,0,0.153
15615,tcp,-,REQ,4,0,180,0,0.153
8414,tcp,-,REQ,6,0,270,0,0.152
26512,tcp,-,REQ,6,0,270,0,0.152
2484,tcp,-,REQ,6,0,270,0,0.152
6558,tcp,-,REQ,6,0,270,0,0.152
7301,tcp,-,REQ,6,0,270,0,0.152
7080,tcp,-,REQ,6,0,270,0,0.152


Before revealing labels, read the top of the list like a hunter. What kinds of flows are these? Would you open a ticket for them?

## Step 2: Reveal the labels and grade the hunt at a fixed budget

An analyst can review maybe 50 to 300 leads in a shift. What matters is how many real attacks land inside that budget.

In [4]:
labels = hunt_week["label"].values
print(f"Attacks hidden in the hunt week: {labels.sum()} of {len(labels)} ({labels.mean():.1%})")
print(f"Ranking quality (ROC AUC): {roc_auc_score(labels, hunt_scores):.3f}")

budget = precision_at_k(labels, hunt_scores)
display(budget.round(3))

top_1000 = hunt_week.sort_values("anomaly_score", ascending=False).head(1000)
print("Attack categories found in the top 1,000:")
display(top_1000.loc[top_1000["label"] == 1, "attack_cat"].value_counts().to_frame("flows"))

Attacks hidden in the hunt week: 300 of 30000 (1.0%)
Ranking quality (ROC AUC): 0.812


,analyst_budget_k,attacks_found,precision_at_k,lift_over_random
0,50,0,0.000,0.000
1,100,0,0.000,0.000
2,300,10,0.033,3.333
3,1000,65,0.065,6.500


Attack categories found in the top 1,000:


,flows
attack_cat,
Generic,61
Exploits,2
DoS,1
Reconnaissance,1


**Unusual is not malicious.** The ranking as a whole is far better than chance, yet the very top of the list is dominated by benign oddities. An analyst who only reviews the top 50 or 100 finds little or nothing. Anomaly scores are a hunting aid, not an alert queue.

## Step 3: What if the baseline was not clean?

In real networks you rarely know your baseline week was attack-free. Retrain "normal" on a window where 30% of flows were attacks, and score the same hunt week.

In [5]:
contaminated = realistic_mix(train, attack_share=0.30, n_rows=30000, random_state=3)
X_cont, _ = split_xy(contaminated, "label")
X_cont = drop_identifier_like_columns(X_cont)[X_base.columns]

contaminated_model = fit_anomaly_model(X_cont)
contaminated_scores = anomaly_scores(contaminated_model, X_hunt)

comparison = pd.DataFrame({
    "baseline": ["clean (0% attacks)", "contaminated (30% attacks)"],
    "roc_auc": [roc_auc_score(labels, hunt_scores), roc_auc_score(labels, contaminated_scores)],
    "attacks_in_top_300": [
        int(labels[np.argsort(-hunt_scores)[:300]].sum()),
        int(labels[np.argsort(-contaminated_scores)[:300]].sum()),
    ],
})
display(comparison.round(3))

,baseline,roc_auc,attacks_in_top_300
0,clean (0% attacks),0.812,10
1,contaminated (30% attacks),0.641,0


When attacks are already inside the baseline, the model learns them as normal, and they stop looking unusual. This is the hunting version of data poisoning (Lab 14).

## Step 4: Cluster the hunt week, then decide where to look

Instead of reading individual flows, group the week into behavior clusters and profile each one. Still no labels.

In [6]:
clusters = cluster_window(X_base, X_hunt, n_clusters=12, random_state=0)
profile = cluster_profile(hunt_week, clusters)
display(profile.round(1))

,flows,top_proto,top_service,top_state,median_spkts,median_dpkts,median_sbytes,median_ct_srv_src
cluster,,,,,,,,
11,291,tcp,ftp,FIN,52.0,54.0,2934.0,1.0
7,368,tcp,-,CON,6.0,2.0,998.0,3.0
5,498,tcp,-,REQ,6.0,0.0,360.0,27.0
4,792,arp,-,INT,1.0,0.0,46.0,1.0
9,845,udp,-,INT,2.0,0.0,1064.0,29.0
8,1315,tcp,http,FIN,12.0,18.0,1580.0,2.0
10,1588,tcp,-,FIN,16.0,18.0,1540.0,10.5
2,1962,tcp,http,FIN,10.0,8.0,824.0,3.0
6,3128,udp,-,INT,2.0,0.0,104.0,5.0


Pick the cluster you would hunt in first, and write down why, *before* running the next cell. Good reasons: conversations that never complete, one-way traffic, unusual protocol and state combinations.

In [7]:
graded = hunt_week.assign(cluster=clusters).groupby("cluster").agg(
    flows=("label", "size"),
    attacks=("label", "sum"),
    attack_share=("label", "mean"),
)
graded["lift_over_base_rate"] = graded["attack_share"] / labels.mean()
display(graded.sort_values("lift_over_base_rate", ascending=False).round(3))

,flows,attacks,attack_share,lift_over_base_rate
cluster,,,,
9,845,106,0.125,12.544
6,3128,88,0.028,2.813
2,1962,26,0.013,1.325
5,498,5,0.010,1.004
11,291,2,0.007,0.687
1,8817,60,0.007,0.681
8,1315,7,0.005,0.532
3,6742,5,0.001,0.074
0,3654,1,0.000,0.027


## Step 5: Turn the best cluster into a hunt hypothesis

A cluster is a lead. A hypothesis is a query you can rerun every week. Write it in plain terms, run it, and measure its lift.

In [8]:
hypothesis = (
    (hunt_week["proto"] == "udp")
    & (hunt_week["state"] == "INT")
    & (hunt_week["dpkts"] == 0)
)
hits = hunt_week[hypothesis]
print("Hypothesis: UDP conversations that never got a reply (state INT, zero packets back)")
print(f"Flows matched: {len(hits)} | attacks among them: {hits['label'].sum()} "
      f"({hits['label'].mean():.1%}, {hits['label'].mean() / labels.mean():.1f}x the base rate)")
display(hits.loc[hits["label"] == 1, "attack_cat"].value_counts().to_frame("attack flows"))

Hypothesis: UDP conversations that never got a reply (state INT, zero packets back)
Flows matched: 2996 | attacks among them: 139 (4.6%, 4.6x the base rate)


,attack flows
attack_cat,
Generic,114
Reconnaissance,14
Fuzzers,9
Shellcode,1
Exploits,1


## Your analysis

1. Describe the flows at the top of the anomaly list. Why are they unusual, and why are they mostly benign?
2. At a budget of 300 reviews, which approach found more attacks: top anomalies or the best cluster?
3. How would you check that a baseline week is clean enough to learn from?
4. Your hypothesis has a lift over the base rate. What would it miss?
5. Which other telemetry (DNS, endpoint, authentication) would sharpen this hunt?

**Takeaway:** unsupervised methods do not replace labels; they decide where a human should look. Clean baselines, a realistic base rate, and a fixed review budget decide whether a hunt is useful.

## From lab to SOC

Threat hunters use anomaly scores and clusters to generate leads, then turn productive leads into repeatable queries and, eventually, detections. The metric that matters is attacks found per hour of analyst time, measured at a realistic base rate.

**Check yourself**
1. Why does anomaly detection fail on UNSW-NB15's raw test set, where most rows are attacks?
2. A new backup tool appears and dominates your anomaly list for a week. What do you do?
3. Turn this lab's hypothesis into a weekly hunt: what would you schedule, and what would you measure?